In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset

from google.colab import userdata
from huggingface_hub import login
import wandb

login(userdata.get('KEY_HAGGINGFACE'))
import wandb

wandb.login(userdata.get('KEY_WANDB'))

/usr/local/lib/python3.13/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: bouzid-ghassen (bouzid-ghassen-digiwaves) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [4]:

model_name = "Qwen/Qwen2.5-1.5B-Instruct"
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype="auto",
)
tokenizer = AutoTokenizer.from_pretrained(model_name)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

#Zero-shot vs. CoT Few-shot Inference

In [5]:
messages = [
    {"role": "system", "content": "You are an expert Python programmer."},
    {"role": "user", "content":"""Task:
Write a function to find the first character that appears consecutively twice in a string. If no character appears consecutively twice, return None.

Tests:
assert first_consecutive_dup("giza-pyramids") == None
assert first_consecutive_dup("abcceedde") == "c"
assert first_consecutive_dup("aba") == None"""}
]

prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors='pt').to(model.device)

outputs = model.generate(**inputs, max_new_tokens=1024, do_sample=False)
outputs = outputs[0][inputs["input_ids"].shape[1]:]
print(tokenizer.decode(outputs, skip_special_tokens=True))


To solve this problem, we can iterate through the string and keep track of the last seen character and its index. If we encounter a character that is different from the last seen one, we check if it has appeared before at any point. Here's how you can implement this:

```python
def first_consecutive_dup(s):
    # Dictionary to store the last occurrence of each character
    char_index = {}
    
    for i, c in enumerate(s):
        # Check if the current character has been seen before
        if c in char_index and char_index[c] >= 0:
            # Return the character that was previously seen after the current position
            return s[char_index[c]:i+1]
        
        # Update the last occurrence of the current character
        char_index[c] = i
    
    # If no consecutive duplicate characters were found, return None
    return None

# Test cases
assert first_consecutive_dup("giza-pyramids") == None
assert first_consecutive_dup("abcceedde") == "c"
assert first_consecutive_dup

In [6]:
messages = [
    {"role": "system", "content": "You are an expert Python programmer."},
    {"role": "user", "content":""" Task:
Write a function to find the maximum difference between any two adjacent elements in a list.
Tests:
assert max_adjacent_diff([1, 5, 2, 10]) == 8

Thinking:
1. We need to look at pairs of elements that are right next to each other.
2. Pair 1: 1 and 5 (difference is |1 - 5| = 4).
3. Pair 2: 5 and 2 (difference is |5 - 2| = 3).
4. Pair 3: 2 and 10 (difference is |2 - 10| = 8).
5. The largest difference tracked is 8.
Code:
```python
def max_adjacent_diff(lst):
    max_diff = 0
    for i in range(len(lst) - 1):
        diff = abs(lst[i] - lst[i+1])
        if diff > max_diff:
            max_diff = diff
    return max_diff

assert max_adjacent_diff([1, 5, 2, 10]) == 8
```

Task:
Write a function to find the first character that appears consecutively twice in a string. If no character appears consecutively twice, return None.

Tests:
assert first_consecutive_dup("giza-pyramids") == None
assert first_consecutive_dup("abcceedde") == "c"
assert first_consecutive_dup("aba") == None"""}
]

prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors='pt').to(model.device)

outputs = model.generate(**inputs, max_new_tokens=1024, do_sample=False)
outputs = outputs[0][inputs["input_ids"].shape[1]:]
print(tokenizer.decode(outputs, skip_special_tokens=True))


To solve this task, we can iterate through the string while keeping track of the last seen character and its index. If we encounter a character that matches the current one but was seen before, it means we have found our answer. Here's how you can implement this:

```python
def first_consecutive_dup(s):
    # Initialize variables to keep track of the last seen character and its index
    last_seen_char = None
    last_seen_index = -1
    
    # Iterate through the string starting from the second character
    for i in range(1, len(s)):
        char = s[i]
        
        # Check if the current character has been seen before
        if char == last_seen_char:
            return char
        
        # Update the last seen character and its index
        last_seen_char = char
        last_seen_index = i
    
    # Return None if no consecutive duplicate characters were found
    return None

# Test cases
assert first_consecutive_dup("giza-pyramids") == None
assert first_consecutive_dup(

#Dataset

In [7]:
dataset = load_dataset("moazeldegwy/MBPP-Reasoning-Hybrid", split="train")
dataset

README.md:   0%|          | 0.00/737 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  742kB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  191kB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/1010 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/234 [00:00<?, ? examples/s]

Dataset({
    features: ['task_id', 'distilled_from', 'split', 'instruction', 'code', 'test_list', 'mode', 'conversations', 'n_tokens'],
    num_rows: 1010
})

In [8]:
print("User:\n",dataset[0]["conversations"][0]['value'])
print("Asstant:\n",dataset[0]["conversations"][1]['value'])
print(20*"==")
print("User:\n",dataset[1]["conversations"][0]['value'])
print("Asstant:\n",dataset[1]["conversations"][1]['value'])

User:
 Task:
You are an expert Python programmer.
Write a function to reverse words seperated by spaces in a given string.

Test:
assert reverse_words("python program")==("program python")
assert reverse_words("java language")==("language java")
assert reverse_words("indian man")==("man indian")
Asstant:
 <think>
Okay, I need to write a Python function called reverse_words that takes a string and returns a new string where the words are reversed. Let me think about how to approach this.

The problem says to reverse the words in the string. For example, "python program" becomes "program python". So the steps might be: split the string into individual words, reverse the list of words, then join them back with spaces.

First, I should split the input string into words. In Python, the split() method without arguments splits on whitespace, which works here. Then, reverse the list. But how to reverse? Maybe using slicing [::-1]. Then join with spaces.

So the function would be something like